In [ ]:
#!/usr/bin/env python3
import json
import os
import sys
import psycopg2
from psycopg2.extras import Json

# ---------- CONFIG ----------
DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = int(os.getenv("PGPORT", "5432"))
DB_NAME = os.getenv("PGDATABASE", "postgres")
DB_USER = os.getenv("PGUSER", "postgres")
DB_PASS = os.getenv("PGPASSWORD", "postgres")
INPUT_FILE = "combined.json"   # change if needed
# ----------------------------

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_to_key_map(obj):
    """
    Accepts:
      - a dict mapping keyword->data  (your sample)
      - a list of dicts where each element is a mapping keyword->data
      - a list of dicts where each element is { keyword: { years:..., countries:..., authors:... } }
    Returns: dict mapping keyword -> data
    """
    out = {}
    if isinstance(obj, dict):
        # assume top-level mapping: keyword -> data
        return obj
    elif isinstance(obj, list):
        # combine entries
        for element in obj:
            if isinstance(element, dict):
                # element might be like {"vle curves": {...}, "co2": {...}}
                for k, v in element.items():
                    if k in out:
                        # if duplicate keys across files, we will keep both by merging shallowly;
                        # final merge/upsert behavior is controlled in DB (see upsert below)
                        out[k] = merge_jsonb_shallow(out[k], v)
                    else:
                        out[k] = v
            else:
                raise ValueError("Unexpected element type in array: " + str(type(element)))
        return out
    else:
        raise ValueError("Unsupported top-level JSON type: " + str(type(obj)))

def merge_jsonb_shallow(a, b):
    """
    Simple shallow merge for Python dicts: keys from b overwrite keys from a.
    This is only used to combine duplicate keys while reading multiple files.
    If you don't want any merging here, you can change logic to keep the first or last.
    """
    if not isinstance(a, dict):
        return b
    if not isinstance(b, dict):
        return a
    merged = dict(a)
    for k, v in b.items():
        if k in merged and isinstance(merged[k], dict) and isinstance(v, dict):
            # merge nested dicts shallowly
            nested = dict(merged[k])
            nested.update(v)
            merged[k] = nested
        else:
            merged[k] = v
    return merged

def upsert_keyword(conn, keyword, years, countries, authors):
    """
    Upsert: insert new row or update existing by merging JSON objects (concatenate / overwrite semantics).
    Note: jsonb || jsonb merges objects; keys from RHS override LHS on conflicts for same key.
    """
    sql = """
    INSERT INTO keyword_stats (keyword, years, countries, authors)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (keyword) DO UPDATE
      SET years = keyword_stats.years || EXCLUDED.years,
          countries = keyword_stats.countries || EXCLUDED.countries,
          authors = keyword_stats.authors || EXCLUDED.authors,
          updated_at = now()
    ;
    """
    with conn.cursor() as cur:
        cur.execute(sql, (keyword, Json(years), Json(countries), Json(authors)))
    conn.commit()

def main():
    if not os.path.exists(INPUT_FILE):
        print("Input file not found:", INPUT_FILE, file=sys.stderr)
        sys.exit(1)

    data = load_json_file(INPUT_FILE)
    key_map = normalize_to_key_map(data)  # map: keyword -> { years:..., countries:..., authors:... }

    print(f"Found {len(key_map)} keywords to insert/upsert.")

    conn = psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASS
    )

    try:
        for k, v in key_map.items():
            # v is expected like { "years": {...}, "countries": {...}, "authors": {...} }
            years = v.get("years") or {}
            countries = v.get("countries") or {}
            authors = v.get("authors") or {}
            # Defensive: ensure types are dicts
            if not isinstance(years, dict) or not isinstance(countries, dict) or not isinstance(authors, dict):
                print(f"Skipping {k}: unexpected structure (years/countries/authors must be objects).", file=sys.stderr)
                continue
            upsert_keyword(conn, k, years, countries, authors)
            print("Upserted:", k)
    finally:
        conn.close()

if __name__ == "__main__":
    main()


In [1]:
!pip install psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.7 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 14.3 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

os.environ["PGHOST"] = "localhost"
os.environ["PGPORT"] = "5432"
os.environ["PGDATABASE"] = "mydb"
os.environ["PGUSER"] = "postgres"
os.environ["PGPASSWORD"] = "mysecretpassword"


In [4]:
import json, os, sys
import psycopg2
from psycopg2.extras import Json

DB = {
  "host": os.getenv("PGHOST","localhost"),
  "port": int(os.getenv("PGPORT","5432")),
  "dbname": os.getenv("PGDATABASE","mydb"),
  "user": os.getenv("PGUSER","postgres"),
  "password": os.getenv("PGPASSWORD","mysecretpassword")
}

INPUT_FILE = "keyword_aggregate_all_results.json"

def load_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)

def normalize(obj):
    out = {}
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        for element in obj:
            if not isinstance(element, dict):
                raise ValueError("array element not an object")
            for k,v in element.items():
                out[k] = v
        return out
    raise ValueError("unsupported JSON top-level")

def upsert(conn, keyword, years, countries, authors):
    sql = """
    INSERT INTO keyword_stats (keyword, years, countries, authors)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (keyword) DO UPDATE
      SET years = keyword_stats.years || EXCLUDED.years,
          countries = keyword_stats.countries || EXCLUDED.countries,
          authors = keyword_stats.authors || EXCLUDED.authors,
          updated_at = now();
    """
    with conn.cursor() as cur:
        cur.execute(sql, (keyword, Json(years), Json(countries), Json(authors)))
    conn.commit()

def main():
    raw = load_json(INPUT_FILE)
    mapping = normalize(raw)

    conn = psycopg2.connect(**DB)

    try:
        for k,v in mapping.items():
            yrs = v.get("years", {}) or {}
            cts = v.get("countries", {}) or {}
            auth = v.get("authors", {}) or {}

            upsert(conn, k, yrs, cts, auth)
            print("Upserted:", k)

    finally:
        conn.close()

main()


OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"


In [6]:
# install driver (if not already)
!pip install -q psycopg2-binary

import os, psycopg2

# set env vars for the notebook session
os.environ["PGHOST"] = "127.0.0.1"   # force IPv4
os.environ["PGPORT"] = "5432"
os.environ["PGDATABASE"] = "mydb"
os.environ["PGUSER"] = "pguser"
os.environ["PGPASSWORD"] = "mysecretpassword"

DB = {
    "host": os.getenv("PGHOST"),
    "port": int(os.getenv("PGPORT")),
    "dbname": os.getenv("PGDATABASE"),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}

try:
    conn = psycopg2.connect(**DB)
    print("Connected OK — server version:", conn.server_version)
    conn.close()
except Exception as e:
    print("Connection failed:", type(e).__name__, e)


Connected OK — server version: 150015



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Load / upsert combined.json into keyword_stats (notebook-friendly)
import json, os, sys
import psycopg2
from psycopg2.extras import Json

# --- DB config from env (already set in previous cell) ---
DB = {
    "host": os.getenv("PGHOST","127.0.0.1"),
    "port": int(os.getenv("PGPORT","5432")),
    "dbname": os.getenv("PGDATABASE","mydb"),
    "user": os.getenv("PGUSER","pguser"),
    "password": os.getenv("PGPASSWORD","mysecretpassword"),
}

INPUT_FILE = "keyword_aggregate_all_results.json"   # change to full path if needed

# --- Helpers ---
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_top(obj):
    """
    Accept either:
     - top-level dict: { keyword: {...}, ... }
     - top-level list of dicts: [ {keyword: {...}}, {keyword2: {...}}, ... ]
    Returns dict mapping keyword -> data (last-wins on duplicate keys).
    """
    out = {}
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        for element in obj:
            if not isinstance(element, dict):
                raise ValueError("Array element is not an object.")
            for k,v in element.items():
                out[k] = v
        return out
    raise ValueError("Unsupported top-level JSON type: " + str(type(obj)))

def upsert(conn, keyword, years, countries, authors):
    sql = """
    INSERT INTO keyword_stats (keyword, years, countries, authors)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (keyword) DO UPDATE
      SET years = keyword_stats.years || EXCLUDED.years,
          countries = keyword_stats.countries || EXCLUDED.countries,
          authors = keyword_stats.authors || EXCLUDED.authors,
          updated_at = now();
    """
    with conn.cursor() as cur:
        cur.execute(sql, (keyword, Json(years), Json(countries), Json(authors)))

# --- Main loader ---
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file not found in current dir: {INPUT_FILE}\nCurrent dir: {os.getcwd()}\nFiles: {os.listdir()}")

data = load_json(INPUT_FILE)
mapping = normalize_top(data)
print(f"Preparing to upsert {len(mapping)} keywords...")

conn = psycopg2.connect(**DB)
try:
    with conn:
        with conn.cursor() as _:
            # iterate and upsert; commit done by context manager
            count = 0
            for k, v in mapping.items():
                yrs = v.get("years", {}) or {}
                cts = v.get("countries", {}) or {}
                auth = v.get("authors", {}) or {}

                # defensive checks
                if not (isinstance(yrs, dict) and isinstance(cts, dict) and isinstance(auth, dict)):
                    print(f"Skipping {k!r}: invalid structure (years/countries/authors must be objects).")
                    continue

                upsert(conn, k, yrs, cts, auth)
                count += 1
                if count % 50 == 0:
                    print(f"Upserted {count}...")

    print(f"Done. Upserted {count} keywords.")
finally:
    conn.close()


Preparing to upsert 11213 keywords...
Upserted 50...
Upserted 100...
Upserted 150...
Upserted 200...
Upserted 250...
Upserted 300...
Upserted 350...
Upserted 400...
Upserted 450...
Upserted 500...
Upserted 550...
Upserted 600...
Upserted 650...
Upserted 700...
Upserted 750...
Upserted 800...
Upserted 850...
Upserted 900...
Upserted 950...
Upserted 1000...
Upserted 1050...
Upserted 1100...
Upserted 1150...
Upserted 1200...
Upserted 1250...
Upserted 1300...
Upserted 1350...
Upserted 1400...
Upserted 1450...
Upserted 1500...
Upserted 1550...
Upserted 1600...
Upserted 1650...
Upserted 1700...
Upserted 1750...
Upserted 1800...
Upserted 1850...
Upserted 1900...
Upserted 1950...
Upserted 2000...
Upserted 2050...
Upserted 2100...
Upserted 2150...
Upserted 2200...
Upserted 2250...
Upserted 2300...
Upserted 2350...
Upserted 2400...
Upserted 2450...
Upserted 2500...
Upserted 2550...
Upserted 2600...
Upserted 2650...
Upserted 2700...
Upserted 2750...
Upserted 2800...
Upserted 2850...
Upserted 2900